# Bosch jobs classification prototype

This notebook follows the Waymo classification workflow while using the
normalized output from `scrapers/bosch_scraper.py`. It demonstrates:

1. loading and validating Bosch job data;
2. selecting AV-relevant roles;
3. extracting a small, traceable role taxonomy with an LLM;
4. extracting salary ranges only when they appear in a job advertisement;
5. combining classification and salary results for later analysis.

The notebook does not invent missing values. Every result retains its
source job ID and URL.


## 1. Load the normalized Bosch dataset

Run `python scrapers/bosch_scraper.py` from the repository root before
running this notebook. The scraper creates `data/bosch_jobs.json`.


In [1]:
# Install project dependencies once from the repository root:
# pip install -r requirements.txt


In [2]:
import html
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from groq import Groq


def find_project_root() -> Path:
    """Find the repository whether Jupyter starts in root or notebooks/."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "bosch_jobs.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find data/bosch_jobs.json. Run "
        "'python scrapers/bosch_scraper.py' first."
    )


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "bosch_jobs.json"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

# Support either a repository-root .env or notebooks/.env.
load_dotenv(PROJECT_ROOT / ".env")
load_dotenv(NOTEBOOK_DIR / ".env", override=False)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input data:   {DATA_PATH}")


Project root: C:\Users\Harshil\Documents\projects\Autonomous_Vehicle_Job_Profiles_Group3
Input data:   C:\Users\Harshil\Documents\projects\Autonomous_Vehicle_Job_Profiles_Group3\data\bosch_jobs.json


In [3]:
with DATA_PATH.open("r", encoding="utf-8") as file:
    job_records = json.load(file)

if not isinstance(job_records, list) or not job_records:
    raise ValueError("Bosch JSON must contain a non-empty list of jobs.")

jobs_df = pd.DataFrame(job_records)
required_columns = {
    "company",
    "job_id",
    "job_title",
    "location",
    "description",
    "source_url",
    "posting_date",
}
missing_columns = sorted(required_columns - set(jobs_df.columns))
if missing_columns:
    raise ValueError(f"Bosch data is missing columns: {missing_columns}")

jobs_df["job_id"] = jobs_df["job_id"].astype(str)
duplicate_count = int(jobs_df["job_id"].duplicated().sum())
if duplicate_count:
    raise ValueError(f"Bosch data contains {duplicate_count} duplicate job IDs.")

print(f"Loaded jobs: {len(jobs_df)}")
print(f"Columns: {len(jobs_df.columns)}")
print(f"Duplicate job IDs: {duplicate_count}")


Loaded jobs: 100
Columns: 23
Duplicate job IDs: 0


In [4]:
quality_summary = pd.Series(
    {
        "total_jobs": len(jobs_df),
        "unique_job_ids": jobs_df["job_id"].nunique(),
        "jobs_with_descriptions": jobs_df["description"].ne("").sum(),
        "jobs_with_posting_dates": jobs_df["posting_date"].notna().sum(),
        "distinct_locations": jobs_df["location"].nunique(),
        "distinct_teams": jobs_df["team"].nunique(),
    },
    name="value",
)
quality_summary.to_frame()


,value
total_jobs,100
unique_job_ids,100
jobs_with_descriptions,100
jobs_with_posting_dates,100
distinct_locations,65
distinct_teams,22


In [5]:
display_columns = [
    "job_id",
    "job_title",
    "location",
    "team",
    "commitment",
    "workplace_type",
    "posting_date",
    "source_url",
]
jobs_df[display_columns].head(10)


,job_id,job_title,location,team,commitment,workplace_type,posting_date,source_url
0,744000142793624,Стажер в отдел логистики,"Almaty, Almaty, Kazakhstan",Other,Intern,onsite,2026-08-11T06:30:03.469Z,https://jobs.smartrecruiters.com/BoschGroup/74...
1,744000142795339,Airbag - Senior Developer,"bangalore, , India",Engineering,Full-time,onsite,2026-08-11T06:29:09.039Z,https://jobs.smartrecruiters.com/BoschGroup/74...
2,744000142795329,Field Support Responsible,"İstanbul, , Turkey",Sales,Full-time,hybrid,2026-08-11T06:28:49.524Z,https://jobs.smartrecruiters.com/BoschGroup/74...
3,744000142794959,Pflichtpraktikum in der Fertigungs- und Erzeug...,"Salzgitter, NIEDERSACHSEN, Germany",Project Management,Full-time,onsite,2026-08-11T06:25:15.513Z,https://jobs.smartrecruiters.com/BoschGroup/74...
4,744000142794889,[CTG] Controlling Officer – Reporting and Fina...,"Tân Bình, Hồ Chí Minh, Vietnam",Finance,Full-time,onsite,2026-08-11T06:23:29.266Z,https://jobs.smartrecruiters.com/BoschGroup/74...
5,744000142794729,Field Support Responsible (İzmir),"İzmir, , Turkey",Sales,Full-time,hybrid,2026-08-11T06:22:12.997Z,https://jobs.smartrecruiters.com/BoschGroup/74...
6,744000142794470,Apprenti Contrôleur de gestion industriel H/F/N,"Saint-Thégonnec Loc-Eguiner, Bretagne, France",Administrative,Full-time,onsite,2026-08-11T06:19:58.899Z,https://jobs.smartrecruiters.com/BoschGroup/74...
7,744000142793784,Field Support Responsible (Antalya),"Antalya, , Turkey",Sales,Full-time,hybrid,2026-08-11T06:19:26.290Z,https://jobs.smartrecruiters.com/BoschGroup/74...
8,744000142792704,项目采购工程师_ME,"Suzhou, Jiangsu, China",Purchasing,Full-time,onsite,2026-08-11T06:18:14.915Z,https://jobs.smartrecruiters.com/BoschGroup/74...
9,744000142793490,工艺工程师 Planning Engineer_QinP,"Qingdao, Shandong, China",Engineering,Full-time,onsite,2026-08-11T06:15:36.831Z,https://jobs.smartrecruiters.com/BoschGroup/74...


## 2. Select mobility-relevant roles and define the classification workflow

Bosch is a diversified technology company, so its general company description
mentions mobility even for unrelated jobs. To reduce false positives, this
prototype uses a stricter title-only filter for mobility, embedded systems,
vehicle safety, validation, robotics, and autonomous-driving signals. The LLM
then receives the complete description for each selected role.


In [6]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scrapers.service.job_prefilter import JobPrefilter

prefilter = JobPrefilter.from_config(
    PROJECT_ROOT / "notebooks" / "config" / "job_prefilter.yaml"
)
prefilter_result = prefilter.filter(jobs_df.to_dict(orient="records"))
prefilter_result.write_outputs(PROJECT_ROOT / "data" / "job_prefilter" / "bosch")
included_job_ids = {
    decision.job_id for decision in prefilter_result.decisions if decision.included
}
technical_jobs_df = jobs_df[
    jobs_df["job_id"].astype(str).isin(included_job_ids)
].copy()

print(f"All Bosch jobs in this batch: {prefilter_result.before_count}")
print(f"Jobs sent to the LLM: {prefilter_result.after_count}")
print(f"Jobs retained in the exclusion audit: {len(prefilter_result.excluded)}")
pd.DataFrame(prefilter_result.company_metrics)


All Bosch jobs in this batch: 100
Mobility/AV title candidates: 8


,job_id,job_title,department,location,language
1,744000142795339,Airbag - Senior Developer,,"bangalore, , India",en
18,744000142788699,Airbag - Senior Developer,,"bengaluru, , India",en
22,744000142787799,Pflichtpraktikum in der Organisationsentwicklu...,,"Stuttgart, BW, Germany",de
50,744000142716179,Embedded Software Engineer Jr,,"Guadalajara, Jal., Mexico",en
70,744000142642789,Validation Engineer - eBike (f/m/div.),,"Ovar, , Portugal",en
75,744000142631869,Model-based software engineer for MEMS sensors,,"Budapest, , Hungary",en
76,744000142630194,Beszerzési Minőségbiztosítási Mérnök Gyakornok...,,"Miskolc, , Hungary",hu
84,744000142617299,Pflichtpraktikum im strategischen Supply Chain...,,"Stuttgart, BW, Germany",de


In [7]:
groq_api_key = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
groq_client = Groq(api_key=groq_api_key) if groq_api_key else None

if groq_client is None:
    print(
        "GROQ_API_KEY is not set. Data and salary cells will still run; "
        "LLM classification will be skipped."
    )
else:
    print(f"Groq client ready. Model: {GROQ_MODEL}")


GROQ_API_KEY is not set. Data and salary cells will still run; LLM classification will be skipped.


In [8]:
job_prompt = """You are analyzing one selected Bosch job posting for
an autonomous-vehicle and mobility-skills research project. Bosch is a broad
technology company, so do not assume that every role is related to autonomous
driving. Use ONLY the supplied job data.

Extract:
1. A short role_profile name specific to this job. Preserve non-AV roles as
   their real function rather than forcing them into an AV category.
2. Specific technical skills, tools, platforms, and technologies that are
   explicitly mentioned. Do not infer unmentioned skills.
3. One broad functional_area, such as Mobility / Embedded Systems,
   Autonomous Driving / ADAS, Hardware / Sensors, Validation / Safety,
   Manufacturing / Operations, Data / Cloud, Business / Product, or
   Corporate / Support.

Respond ONLY with valid JSON using exactly this shape:
{
  "company": "Bosch",
  "title": "...",
  "career_page_url": "...",
  "role_profile": "...",
  "skills": ["..."],
  "functional_area": "..."
}
"""


In [9]:
CLASSIFICATION_FIELDS = {
    "company",
    "title",
    "career_page_url",
    "role_profile",
    "skills",
    "functional_area",
}


def parse_llm_json(content: str) -> dict:
    text = (content or "").strip()
    if text.startswith("```"):
        text = text.strip("`").strip()
        if text.lower().startswith("json"):
            text = text[4:].strip()

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end <= start:
            raise
        parsed = json.loads(text[start : end + 1])

    if not isinstance(parsed, dict):
        raise ValueError("LLM response must be a JSON object.")
    missing = sorted(CLASSIFICATION_FIELDS - set(parsed))
    if missing:
        raise ValueError(f"LLM response is missing fields: {missing}")
    if not isinstance(parsed["skills"], list):
        raise ValueError("LLM skills must be a JSON list.")
    return parsed


def classify_job(job: pd.Series) -> dict:
    if groq_client is None:
        raise RuntimeError("Set GROQ_API_KEY before classifying jobs.")

    context = {
        "job_id": job["job_id"],
        "title": job["job_title"],
        "team": job.get("team", ""),
        "location": job["location"],
        "career_page_url": job["source_url"],
        "description": job["description"],
    }
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": job_prompt + "\n\nJOB DATA:\n" + json.dumps(context),
            }
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    parsed = parse_llm_json(response.choices[0].message.content)

    # Preserve source-of-truth identifiers instead of trusting model copies.
    parsed["job_id"] = str(job["job_id"])
    parsed["company"] = "Bosch"
    parsed["title"] = job["job_title"]
    parsed["career_page_url"] = job["source_url"]
    return parsed


In [10]:
SAMPLE_SIZE = 3
sample_source_df = technical_jobs_df
sample_jobs_df = sample_source_df.head(SAMPLE_SIZE).copy()

print(f"Selected {len(sample_jobs_df)} sample jobs dynamically:")
sample_jobs_df[["job_id", "job_title", "location", "source_url"]]


Selected 3 sample jobs dynamically:


,job_id,job_title,location,source_url
1,744000142795339,Airbag - Senior Developer,"bangalore, , India",https://jobs.smartrecruiters.com/BoschGroup/74...
18,744000142788699,Airbag - Senior Developer,"bengaluru, , India",https://jobs.smartrecruiters.com/BoschGroup/74...
22,744000142787799,Pflichtpraktikum in der Organisationsentwicklu...,"Stuttgart, BW, Germany",https://jobs.smartrecruiters.com/BoschGroup/74...


In [11]:
classification_records = []

if groq_client is None:
    print("Classification skipped. Add GROQ_API_KEY to .env and rerun this cell.")
else:
    for _, job in sample_jobs_df.iterrows():
        try:
            classification_records.append(classify_job(job))
            print(f"Classified: {job['job_title']}")
        except Exception as exc:
            print(f"Classification failed for {job['job_id']}: {exc}")
        time.sleep(0.5)

classification_columns = [
    "job_id",
    "company",
    "title",
    "career_page_url",
    "role_profile",
    "skills",
    "functional_area",
]
classification_df = pd.DataFrame(
    classification_records,
    columns=classification_columns,
)
classification_df


Classification skipped. Add GROQ_API_KEY to .env and rerun this cell.


,job_id,company,title,career_page_url,role_profile,skills,functional_area


## 3. Extract salary ranges from Bosch job descriptions

Salary values are extracted only when they appear in the source advertisement.
Bosch postings cover many countries and languages, so this prototype records
only clearly formatted `$`, `£`, or `€` ranges and leaves all other values
missing rather than guessing conversions.


In [12]:
SALARY_RANGE_PATTERN = re.compile(
    r"(?P<symbol>[$£€])\s*"
    r"(?P<minimum>\d{2,3}(?:,\d{3})+(?:\.\d{1,2})?)\s*"
    r"(?:-|–|—|to)\s*"
    r"(?:[$£€]\s*)?"
    r"(?P<maximum>\d{2,3}(?:,\d{3})+(?:\.\d{1,2})?)"
    r"(?:\s*(?P<currency>USD|CAD|GBP|EUR))?",
    flags=re.IGNORECASE,
)
SYMBOL_TO_CURRENCY = {"$": "USD", "£": "GBP", "€": "EUR"}


def extract_salary(description: str) -> pd.Series:
    text = html.unescape(description) if isinstance(description, str) else ""
    match = SALARY_RANGE_PATTERN.search(text)
    if not match:
        return pd.Series(
            {
                "salary_min": None,
                "salary_max": None,
                "salary_currency": None,
                "salary_period": None,
                "salary_snippet": None,
                "salary_source": None,
            }
        )

    snippet_start = max(0, match.start() - 60)
    snippet_end = min(len(text), match.end() + 80)
    context = re.sub(r"\s+", " ", text[snippet_start:snippet_end]).strip()
    period = "hour" if re.search(r"hour|/hr", context, re.I) else "year"

    return pd.Series(
        {
            "salary_min": float(match.group("minimum").replace(",", "")),
            "salary_max": float(match.group("maximum").replace(",", "")),
            "salary_currency": (
                match.group("currency") or SYMBOL_TO_CURRENCY[match.group("symbol")]
            ).upper(),
            "salary_period": period,
            "salary_snippet": context,
            "salary_source": "job_post",
        }
    )


salary_fields_df = jobs_df["description"].apply(extract_salary)
jobs_with_salary_df = pd.concat(
    [jobs_df.reset_index(drop=True), salary_fields_df.reset_index(drop=True)],
    axis=1,
)

salary_report_df = jobs_with_salary_df[
    jobs_with_salary_df["salary_min"].notna()
][
    [
        "job_id",
        "job_title",
        "location",
        "source_url",
        "salary_min",
        "salary_max",
        "salary_currency",
        "salary_period",
        "salary_snippet",
        "salary_source",
    ]
].sort_values(["salary_currency", "salary_min"], ascending=[True, False])

print(f"Total Bosch jobs: {len(jobs_df)}")
print(f"Jobs with salary ranges in the source post: {len(salary_report_df)}")
salary_report_df


Total Bosch jobs: 100
Jobs with salary ranges in the source post: 2


,job_id,job_title,location,source_url,salary_min,salary_max,salary_currency,salary_period,salary_snippet,salary_source
34,744000142742719,Principal Facilities Electrical Engineer (High...,"Roseville, CA, United States",https://jobs.smartrecruiters.com/BoschGroup/74...,150000.0,170000.0,USD,year,e The U.S. base salary range for this full-tim...,job_post
67,744000142649027,SAP Business Process Consultant (Multiple Posi...,"Oakbrook Terrace, IL, United States",https://jobs.smartrecruiters.com/BoschGroup/74...,90709.0,95342.0,USD,year,10% Domestic and International Travel. Salary:...,job_post


## 4. Combine taxonomy and salary results

The merge uses `job_id`, which is retained directly from the scraper.
This keeps every derived field traceable to its original posting.


In [13]:
salary_merge_columns = [
    "job_id",
    "location",
    "salary_min",
    "salary_max",
    "salary_currency",
    "salary_period",
    "salary_snippet",
    "salary_source",
]

if classification_df.empty:
    final_report_df = pd.DataFrame(
        columns=classification_columns
        + [column for column in salary_merge_columns if column != "job_id"]
    )
    print("No classifications to combine yet. Set GROQ_API_KEY and rerun section 2.")
else:
    final_report_df = classification_df.merge(
        jobs_with_salary_df[salary_merge_columns],
        on="job_id",
        how="left",
        validate="one_to_one",
    )
    print(f"Combined rows: {len(final_report_df)}")

final_report_df


No classifications to combine yet. Set GROQ_API_KEY and rerun section 2.


,job_id,company,title,career_page_url,role_profile,skills,functional_area,location,salary_min,salary_max,salary_currency,salary_period,salary_snippet,salary_source


In [14]:
OUTPUT_PATH = PROJECT_ROOT / "data" / "bosch_classification_results.json"

if final_report_df.empty:
    print("Nothing exported because classification results are empty.")
else:
    final_report_df.to_json(
        OUTPUT_PATH,
        orient="records",
        indent=2,
        force_ascii=False,
    )
    print(f"Saved classification results to: {OUTPUT_PATH}")


Nothing exported because classification results are empty.


## 5. Conclusion and next steps

This notebook provides a reproducible Bosch classification prototype:

- scraper output is loaded from a versionable JSON file;
- required fields and duplicate IDs are validated before analysis;
- a strict title filter reduces false AV matches in Bosch's broad job catalog;
- sample jobs are selected dynamically rather than by hard-coded IDs;
- LLM output is parsed and schema-checked before use;
- salary values come only from source advertisements;
- derived results retain job IDs and URLs for traceability.

Next, manually review the selected Bosch candidates and the first three model
outputs. Once the selection rule and taxonomy are accepted, compare results
across Bosch, Waabi, Stack AV, and Waymo, then expand beyond the latest-100
Bosch batch if the sprint scope requires it.
